# Estimating Rare Events in Language Models with Proper Evaluation

This notebook reproduces the paper's pipeline on one (model, distribution) pair:

1. Load a small transformer (`gelu-1l`, `gelu-2l` or `gelu-4l`) and one of the eight input distributions.
2. Load the **ground-truth** output-token probabilities (shipped with this repo in `data/ground_truth/`, derived from 2^32 ground-truth samples).
3. Pick rare target tokens with ground-truth probability in `[1e-9, 1e-5]`.
4. Estimate each target's probability with the baselines **QLD** (bug-corrected), **ITGIS** and **MHIS** (Xu et al. 2024), and with our method **GA-AMLS**.
5. Compare estimates against the ground truth and evaluate all methods with the **Shifted Power Bregman (SPB) loss** under the four scenarios reported in the paper.

Set `QUICK_TEST = True` below for a fast CPU-friendly sanity check (fewer samples/particles; estimates will be noisy). With `QUICK_TEST = False` the exact final configurations from the paper are used (a GPU is strongly recommended).

In [ ]:
import torch as th
import matplotlib.pyplot as plt
from tqdm import tqdm

from ree.methods import QLD, ITGIS, MHIS
from ree.method_utils import (
    N_REPS_DICT,
    RECOMMENDED_TEMPS,
    gen_acts_pre_and_post_ln,
    load_ground_truth_probs,
    pick_random_tokens,
    set_seed,
)
from ree.utils import Transformer, distribution_registry
from ree.amls import GA_AMLS, AMLSConfig
from ree.metrics import SCENARIOS, EPS_CORR, aggregate_spb_loss

set_seed(0)

# Quick sanity-check mode (CPU friendly). Set to False to reproduce the paper's configuration.
QUICK_TEST = False

In [ ]:
# Load the model (downloads weights from HuggingFace on first run)
model_name = "gelu-1l"  # "gelu-1l", "gelu-2l" or "gelu-4l"
device = th.device("cuda" if th.cuda.is_available() else "cpu")
model = Transformer.from_pretrained(model_name).to(device).eval()
print(f"Loaded {model_name} on {device}")

In [ ]:
# Load the ground-truth probabilities saved in this repo and pick rare target tokens
dist_name = "camel"  # one of: hex, camel, indent, ifelse, caps, english, spanish, icl

gt_probs = load_ground_truth_probs(model_name, dist_name, device=str(device))
n_targets = 4 if QUICK_TEST else 16
targets = pick_random_tokens(gt_probs, n_targets, 1e-9, 1e-5)
print("Target tokens:", targets)
print("Ground-truth probabilities:", [f"{gt_probs[t]:.2e}" for t in targets])

In [ ]:
# Generate activation samples of the final-position residual stream:
# - pre-LN activations (before the final layer norm) are used by GA-AMLS
# - post-LN activations (right before the unembedding) are used by QLD
n_act_samples = 2**12 if QUICK_TEST else 2**16
acts_pre_ln, acts_post_ln = gen_acts_pre_and_post_ln(model, dist_name, n_samples=n_act_samples)
print("Activation samples:", acts_pre_ln.shape)

## Generate estimates

- **QLD** uses the post-LN activation samples and the bug-corrected `random_constraint_projection`.
- **ITGIS / MHIS** sample directly in token space using the temperatures recommended by Xu et al. (2024).
- **GA-AMLS** runs MALA-within-adaptive-multilevel-splitting on the whitened pre-LN activations with a Student-T prior. The default `AMLSConfig` is the final configuration used for all experiments in the paper: `n_particles=2000, mh_steps=900, rho=0.6, h=1e-3, target_acc=0.57, burn_percentage=0.14`, Student-T prior, pre-LN score function.

In [ ]:
# Configuration for each method
if QUICK_TEST:
    itgis_kwargs = dict(n_samples=2**9, batch_size=2**5)
    mhis_kwargs = dict(n_samples=2**7, burn_in=2**3, batch_size=16)
    amls_config = AMLSConfig(n_particles=128, mh_steps=30, device=str(device))
else:
    itgis_kwargs = dict(n_samples=2**16)
    mhis_kwargs = dict(n_samples=2**16, burn_in=2**10)
    # Final tuned GA-AMLS configuration from the paper (the defaults)
    amls_config = AMLSConfig(device=str(device))

orig_dists = distribution_registry[dist_name](model.tokenizer, device=model.device).input_dists(
    n_reps=N_REPS_DICT[dist_name]
)

methods = ["QLD", "ITGIS", "MHIS", "GA-AMLS"]
estimates = {method: {} for method in methods}

for target in tqdm(list(targets)):
    estimates["QLD"][target] = QLD(model.unembed.W_U, acts_post_ln, target)
    estimates["ITGIS"][target] = ITGIS(
        model, orig_dists, target,
        temp=RECOMMENDED_TEMPS[model_name]["ITGIS"][dist_name], **itgis_kwargs,
    )
    estimates["MHIS"][target] = MHIS(
        model, orig_dists, target,
        temp=RECOMMENDED_TEMPS[model_name]["MHIS"][dist_name], **mhis_kwargs,
    )
    estimates["GA-AMLS"][target] = GA_AMLS(model, acts_pre_ln, target, config=amls_config)

In [ ]:
# Plot the estimates against the ground truth
plt.figure(figsize=(12, 6))
colors = {"QLD": "#ff7f0e", "ITGIS": "#d62728", "MHIS": "#9467bd", "GA-AMLS": "#1f77b4"}
for method in methods:
    ests = [estimates[method][target] for target in targets]
    plt.scatter(gt_probs[targets].cpu().numpy(), ests, label=method, color=colors[method])

    # Plot the exact zeros at the bottom as crosses
    zero_targets = [target for target in targets if estimates[method][target] == 0]
    plt.scatter(gt_probs[zero_targets].cpu().numpy(), [1e-9] * len(zero_targets),
                color=colors[method], marker="x")

plt.plot([1e-9, 1e-5], [1e-9, 1e-5], label="ground truth", color="black")
plt.xscale("log")
plt.yscale("log")
plt.xlabel("Ground Truth Probability")
plt.ylabel("Estimate")
plt.title(f"Estimates for {dist_name} on {model_name}")
plt.legend()
plt.show()

## Evaluation with the Shifted Power Bregman (SPB) loss

Each estimate \(p\) is scored against the ground truth \(q\) with the weighted shifted power Bregman divergence
\(w(q)\, B_\varepsilon(q \,\|\, p)\) where \(B_\varepsilon(q \,\|\, p) = \int_q^p (t - q)(t + \varepsilon)^{-\alpha}\,dt\) and \(w(q) = q^{\alpha - 2 - \gamma}\).
The shift \(\varepsilon\) is the Corollary-1 value (`EPS_CORR`), calibrated for the benchmarked range \(q \in [10^{-9}, 10^{-5}]\).
We report the average loss over targets for the four \((\alpha, \gamma)\) scenarios used in the paper. **Lower is better.**

In [ ]:
# Average SPB loss per method for the four scenarios from the paper (lower is better)
print(f"Corollary-1 shift eps = {EPS_CORR:.3e}\n")

qs = [float(gt_probs[target]) for target in targets]
spb_losses = {}
for scenario_name, alpha, gamma in SCENARIOS:
    spb_losses[scenario_name] = {}
    for method in methods:
        ps = [estimates[method][target] for target in targets]
        spb_losses[scenario_name][method] = aggregate_spb_loss(qs, ps, alpha=alpha, gamma=gamma)

header = f"{'Scenario (alpha, gamma)':<42}" + "".join(f"{m:>14}" for m in methods)
print(header)
print("-" * len(header))
for scenario_name, alpha, gamma in SCENARIOS:
    label = f"{scenario_name} ({alpha:g}, {gamma:g})"
    row = f"{label:<42}" + "".join(f"{spb_losses[scenario_name][m]:>14.4e}" for m in methods)
    print(row)

In [ ]:
# Bar chart of the average SPB loss (log scale, lower is better)
fig, axes = plt.subplots(1, len(SCENARIOS), figsize=(20, 4.5))
for ax, (scenario_name, alpha, gamma) in zip(axes, SCENARIOS):
    values = [spb_losses[scenario_name][method] for method in methods]
    ax.bar(methods, values, color=[colors[m] for m in methods], alpha=0.85)
    ax.set_yscale("log")
    ax.set_title(f"{scenario_name}\n(alpha={alpha:g}, gamma={gamma:g})", fontsize=10)
    ax.tick_params(axis="x", rotation=20)
axes[0].set_ylabel("Average SPB Loss")
fig.tight_layout()
plt.show()